In [1]:
!pip install tabulate

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor 
import warnings
warnings.filterwarnings('ignore')

# --- C. Dataset Creation & Feature Engineering ---
df = pd.read_csv("DI_Leather_preprocessed.csv")

# Engineer new features strictly from pre-production data
df['theoretical_max_production'] = (df['working_hours'] * 60) / df['total_duration_per_shoe_mins']
df['machine_worker_ratio'] = df['machine_count'] / df['no_of_workers']
df['avg_hours_per_worker'] = df['working_hours'] / df['no_of_workers']

# --- E. Chronological 80/20 Split ---
# Sort chronologically to prevent temporal data leakage. 
# A random split allows the model to "peek" at future production efficiencies and seasonalities.
df['week_start_date'] = pd.to_datetime(df['week_start_date'])
df = df.sort_values('week_start_date').reset_index(drop=True)

# Define target and drop excluded/leakage columns
target = 'actual_qty'
drop_cols = [
    'planned_qty', 'production_gap_qty', 'bottleneck_flag', 
    'machine_capacity_per_day', 'week_start_date', 'week_end_date', 
    'financial_year'
]

X_full = df.drop(columns=[target] + drop_cols)
y_full = df[target]

# Time-based Split
split_idx = int(len(df) * 0.8)
X_train, X_test = X_full.iloc[:split_idx], X_full.iloc[split_idx:]
y_train, y_test = y_full.iloc[:split_idx], y_full.iloc[split_idx:]

# --- D. Preprocessing Pipeline ---
# Categorical columns for One-Hot Encoding (Low cardinality)
categorical_ohe = ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']

# Categorical columns for Target Encoding (High cardinality - 370 unique item names)
# Note: Fitted inside the pipeline to ensure it only learns from the training fold.
categorical_target = ['item_name']

# Numerical columns for Scaling
numerical_cols = [col for col in X_train.columns if col not in categorical_ohe + categorical_target]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_ohe),
        ('cat_te', TargetEncoder(target_type='continuous'), categorical_target) 
    ])

# --- F. Training Multiple Regression Models ---
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1)
}

results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    # Metrics
    results.append({
        'Model': name,
        'Train R²': round(r2_score(y_train, y_train_pred), 3),
        'Test R²': round(r2_score(y_test, y_test_pred), 3),
        'Train MAE': round(mean_absolute_error(y_train, y_train_pred), 3),
        'Test MAE': round(mean_absolute_error(y_test, y_test_pred), 3),
        'Train RMSE': round(np.sqrt(mean_squared_error(y_train, y_train_pred)), 3),
        'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_test_pred)), 3)
    })

results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

| Model             |   Train R² |   Test R² |   Train MAE |   Test MAE |   Train RMSE |   Test RMSE |
|:------------------|-----------:|----------:|------------:|-----------:|-------------:|------------:|
| Linear Regression |      0.891 |     0.887 |       3.371 |      3.482 |        4.298 |       4.42  |
| Ridge Regression  |      0.891 |     0.887 |       3.371 |      3.473 |        4.297 |       4.412 |
| Lasso Regression  |      0.887 |     0.882 |       3.427 |      3.539 |        4.374 |       4.511 |
| Random Forest     |      0.978 |     0.884 |       1.489 |      3.497 |        1.94  |       4.47  |
| Gradient Boosting |      0.906 |     0.89  |       3.13  |      3.409 |        3.99  |       4.351 |
| Extra Trees       |      0.999 |     0.881 |       0.182 |      3.564 |        0.341 |       4.528 |
| XGBoost           |      0.962 |     0.874 |       1.941 |      3.642 |        2.539 |       4.666 |
